# CropSight AI — Browser-Based Tomato Model Training

Train a real YOLO model in Google Colab without installing Python or using a local terminal. This notebook expects the converted `CropSight_tomatOD_YOLO.zip` dataset.

**Important:** the dataset's ripeness labels are a proxy for harvest readiness, not agronomic ground truth. The dataset is CC BY-NC-SA 4.0, so follow its attribution, share-alike and non-commercial terms. Training can take time; enable a GPU in **Runtime → Change runtime type → T4 GPU** if available.

When finished, the notebook downloads `cropsight-model-output.zip`, containing `best.pt` and training metrics. This does not automatically deploy a backend or change the live GitHub Pages website.


## 1. Install training dependencies


In [ ]:
%pip -q install "ultralytics>=8.3,<9.0" pyyaml


## 2. Upload your dataset ZIP
Download `CropSight_tomatOD_YOLO.zip` from the ChatGPT conversation first, then upload it when prompted below.


In [ ]:
from google.colab import files
uploaded = files.upload()  # Choose CropSight_tomatOD_YOLO.zip
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if not zip_names:
    raise ValueError('Please upload the CropSight_tomatOD_YOLO.zip file.')
ZIP_PATH = '/content/' + zip_names[0]
print('Dataset ZIP:', ZIP_PATH)


## 3. Extract and validate the dataset


In [ ]:
from pathlib import Path
import zipfile, yaml, os

DATA_ROOT = Path('/content/cropsight_dataset')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(DATA_ROOT)
yaml_candidates = list(DATA_ROOT.rglob('dataset.yaml'))
if not yaml_candidates:
    raise FileNotFoundError('Could not find dataset.yaml inside the ZIP. Check that you uploaded the converted YOLO dataset.')
DATA_YAML = yaml_candidates[0].resolve()
config = yaml.safe_load(DATA_YAML.read_text(encoding='utf-8'))
DATA_DIR = DATA_YAML.parent
config['path'] = str(DATA_DIR)
config['train'] = 'images/train'
config['val'] = 'images/val'
if (DATA_DIR / 'images/test').exists():
    config['test'] = 'images/test'
expected = {0:'harvest_ready', 1:'nearly_ready', 2:'developing'}
names = config.get('names', {})
if isinstance(names, list): names = {i:n for i,n in enumerate(names)}
config['names'] = names
if len(names) != 3:
    raise ValueError(f'Expected 3 maturity classes, found {names}.')
if any(not (DATA_DIR / 'images' / split).is_dir() for split in ['train','val']):
    raise FileNotFoundError('Dataset must contain images/train and images/val folders.')
for split in ['train','val']:
    image_count = sum(1 for p in (DATA_DIR / 'images' / split).iterdir() if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'})
    label_dir = DATA_DIR / 'labels' / split
    label_count = sum(1 for p in label_dir.glob('*.txt')) if label_dir.exists() else 0
    print(f'{split}: {image_count} images, {label_count} label files')
config_path = Path('/content/cropsight_dataset.yaml')
config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
print('\nUsing dataset YAML:', config_path)
print('Class names:', config['names'])
print('Dataset root:', DATA_DIR)


## 4. Train the model
The default is 50 epochs with YOLO11n. If you hit GPU memory errors, lower `BATCH` to 4 or 2. Results depend on data quality and training time.


In [ ]:
from ultralytics import YOLO
import torch

EPOCHS = 50
IMAGE_SIZE = 640
BATCH = 8 if torch.cuda.is_available() else 4
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
model = YOLO('yolo11n.pt')
train_results = model.train(
    data=str(config_path), epochs=EPOCHS, imgsz=IMAGE_SIZE, batch=BATCH,
    device=DEVICE, project='/content/cropsight-runs', name='tomato-maturity',
    patience=12, workers=2, plots=True, exist_ok=True
)
RUN_DIR = Path(train_results.save_dir)
BEST_PT = RUN_DIR / 'weights' / 'best.pt'
if not BEST_PT.exists():
    raise FileNotFoundError(f'Training did not produce best.pt in {RUN_DIR}')
print('Best model:', BEST_PT)


## 5. Evaluate on the validation split
Review precision, recall and mAP. These numbers do not guarantee reliable harvest decisions in real fields; test on additional images from your own farm before relying on the result.


In [ ]:
trained = YOLO(str(BEST_PT))
metrics = trained.val(data=str(config_path), split='val', imgsz=IMAGE_SIZE, device=DEVICE)
print('Validation metrics:', metrics.results_dict)


## 6. Package and download the model
The ZIP includes the trained weights and available training outputs. Keep the original dataset license and attribution when redistributing dataset-derived materials.


In [ ]:
import shutil
from google.colab import files
OUT_DIR = Path('/content/cropsight-model-output')
OUT_DIR.mkdir(exist_ok=True)
shutil.copy2(BEST_PT, OUT_DIR / 'best.pt')
for name in ['results.csv', 'results.png', 'confusion_matrix.png', 'BoxPR_curve.png', 'BoxF1_curve.png']:
    candidate = RUN_DIR / name
    if candidate.exists(): shutil.copy2(candidate, OUT_DIR / name)
(OUT_DIR / 'MODEL_NOTES.txt').write_text(
    'CropSight AI tomato maturity model\n'
    'Class IDs must match the API: 0=harvest_ready, 1=nearly_ready, 2=developing.\n'
    'Dataset source: tomatOD / DatasetNinja; license CC BY-NC-SA 4.0.\n'
    'Ripeness classes are only a proxy for harvest readiness. Validate on representative field images.\n',
    encoding='utf-8')
archive = shutil.make_archive('/content/cropsight-model-output', 'zip', OUT_DIR)
print('Output ZIP:', archive)
files.download(archive)


## What happens next?
1. After downloading, keep `best.pt` private until you decide where to host the inference API.
2. The GitHub Pages site is static; it cannot run this Python model by itself. To enable real scans on the live site, the model must be deployed behind a reachable API endpoint and `VITE_CROPSIGHT_API_URL` must point to that endpoint.
3. Do not claim the model is production-ready until it is tested on unseen tomato images and its metrics and class mapping are checked.
